# BCEWithLogitsLoss with Invalid Targets


Minimal sanity check for `torch.nn.BCEWithLogitsLoss`.

This notebook shows what happens when a target is `-1` instead of a valid binary label.
The point is not that `-1` is acceptable, but that the loss can still return a number.



[BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) uses:

$ \text{loss} = -\left( y \cdot \log(\sigma(x)) + (1 - y) \cdot \log(1 - \sigma(x)) \right)
$


If a label of `-1` is passed, then: $ y = -1 $
Plug it into the formula:

$ \text{loss} = -\left( -1 \cdot \log(\sigma(x)) + (1 - (-1)) \cdot \log(1 - \sigma(x)) \right)
$

Simplify:


$
\text{loss} = -\left( -\log(\sigma(x)) + 2 \cdot \log(1 - \sigma(x)) \right)
$

$
\text{loss} = \log(\sigma(x)) - 2 \log(1 - \sigma(x)) \quad \text{(not meaningful as a BCE loss)}
$


In [ ]:
# Compare an invalid target against the corrected binary target.

import torch
from torch import nn

criterion = nn.BCEWithLogitsLoss()
preds = torch.tensor([1., -1., 1., -1.]) # These can be any real-valued logits.
labels = torch.tensor([1., -1., 1., 1.]) # Invalid labels with -1.
print("Minus one label:", criterion(preds, labels))
all_correct = torch.tensor([1., 0., 1., 0.])
print("All correct labels: ",criterion(preds, all_correct))

Minus one label: tensor(0.3133)
All correct labels:  tensor(0.3133)


The results are equal!

Maybe PyTorch handles the `-1` label properly?


Test it directly:

In [ ]:
# Invalid target on purpose.
criterion = nn.BCEWithLogitsLoss()


preds = torch.tensor([-1.])
labels = torch.tensor([-1.])

print("Wrong label(-1:", criterion(preds, labels))

tensor(-0.6867)


In [ ]:
# Same logit, valid target.
criterion = nn.BCEWithLogitsLoss()


preds = torch.tensor([-1.])
labels = torch.tensor([0.])

print("Acceptable label(0):", criterion(preds, labels))

tensor(0.3133)


For one prediction, the results are different.

Let's test our bad label vector against another logit vector. For convenience, we create a helper function with two label vectors inside.

In [ ]:
import torch
from torch import nn


criterion = nn.BCEWithLogitsLoss()

def test(pred):
  # Compare invalid and corrected targets for the same logits.
  pred = torch.tensor(pred)
  labels = torch.tensor([1., -1., 1., 1.]) # Invalid labels with -1.
  correct_labels = torch.tensor([1., 0., 1., 1.]) # Valid labels.
  print("Minus one label:", criterion(pred, labels), " vs ", criterion(pred, correct_labels))


test([1., -1., 1., -1.])
test([1., -2., 1., -1.])


Minus one label: tensor(0.3133)  vs  tensor(0.5633)
Minus one label: tensor(0.0167)  vs  tensor(0.5167)


Now the results are different. Check:

In [ ]:
# Correct target for the same logits.
corr_labels = torch.tensor([1., 0, 1., 1.])
print(criterion(preds, corr_labels))

tensor(0.5633)


In [ ]:
# All-positive target for the same logits.
all_ones = torch.tensor([1., 1., 1., 1.])
print(criterion(preds, all_ones))

tensor(0.8133)


In [ ]:
# Same data as the first example, but with valid labels.
all_correct = torch.tensor([1., 0., 1., 0.])
print(criterion(preds, all_correct))

tensor(0.3133)


But why do we have the same loss value (0.3133) in the first example?

It is just a coincidence caused by averaging.

For `preds = [1, -1, 1, -1]`:

With invalid labels `[1, -1, 1, 1]`, the per-element losses are approximately:

$
[0.3133,\ -0.6867,\ 0.3133,\ 1.3133]
$

For the first element, `x = 1`, `y = 1`:

$
-\log(\sigma(1)) \approx 0.3133
$

For the second element, `x = -1`, `y = -1`:

$
\log(\sigma(-1)) - 2\log(1 - \sigma(-1)) \approx -0.6867
$

...

Their mean is:

$
\frac{0.3133 - 0.6867 + 0.3133 + 1.3133}{4} = 0.3133
$

With valid labels `[1, 0, 1, 0]`, the per-element losses are:

$
[0.3133,\ 0.3133,\ 0.3133,\ 0.3133]
$

Their mean is also `0.3133`.

So the first result does not prove that `-1` is a valid target. The negative loss at the second element is cancelled by the larger loss at the fourth element. This cancellation happens only because of this particular set of logits and labels.

Conclusion

`BCEWithLogitsLoss` expects targets in `[0, 1]`. A target of `-1` is invalid, but PyTorch may still compute a numeric loss, so this notebook is only a warning example.
